# Hito 1 - Notebook 01: Comprension del Negocio (Business Understanding)
## Fase 1 de CRISP-DM

**Proyecto:** ALDIMI Core AI - Ecosistema de Gestion Inteligente para el Albergue Divina Misericordia (ALDIMI 2.0).

Este notebook documenta la **Fase 1 de CRISP-DM**: el entendimiento del negocio, la definicion de objetivos de Machine Learning y los criterios de exito, alineados con los Objetivos de Desarrollo Sostenible (ODS).

## 1.1. Resumen Ejecutivo

La Asociacion de Voluntariado de Infancia y Familia (**ALDIMI**), fundada en 2004, brinda atencion integral y gratuita a ninos y adolescentes con cancer en situacion de extrema pobreza. Enfrenta hoy su mayor desafio: la transicion hacia **ALDIMI 2.0**, que **duplicara su capacidad de 50 a 100 familias**.

A esta escala, los procesos manuales resultan ineficientes y peligrosos: un quiebre de stock de medicamentos oncologicos o una priorizacion clinica tardia comprometen la continuidad del tratamiento. El proyecto actua como el **cerebro predictivo** del albergue, transformando una gestion reactiva en preventiva.

## 1.2. Objetivos de Negocio y de Machine Learning

### Objetivo de Negocio 1 - Abastecimiento preventivo (Frente 1: Logistica)
Anticipar el consumo y disponibilidad de insumos criticos en horizontes de **7 y 14 dias** para evitar quiebres de stock durante la expansion 50 -> 100 familias.

- **Tarea de ML:** Regresion (supervisada / series temporales).
- **Variable objetivo:** `Demanda_Fut_7d` y `Demanda_Fut_14d` (demanda/consumo acumulado por insumo en los proximos 7 y 14 dias). El stock proyectado y las alertas de reposicion se **derivan** de la demanda predicha (`Stock_Proyectado = Stock_Actual - Demanda_Predicha`). Se predice la demanda porque el nivel absoluto de stock a ese horizonte no es predecible (sin autocorrelacion), mientras que el consumo si lo es.

### Objetivo de Negocio 2 - Priorizacion clinica (Frente 2: Salud)
Clasificar el **Nivel de Prioridad de Atencion** (Bajo / Medio / Alto) de cada paciente **pediatrico-juvenil** (cohorte Age < 25) combinando factores clinicos y socioeconomicos, como herramienta de apoyo (no de diagnostico).

- **Tarea de ML:** Clasificacion supervisada multiclase.
- **Poblacion:** pacientes con `Age < 25` (alineado con ALDIMI: ninos y adolescentes con cancer).
- **Variable objetivo:** `Prioridad_Atencion`.

## 1.3. Criterios de Exito

| Frente | Metrica principal | Criterio de exito |
|---|---|---|
| Logistica (regresion) | MAE / RMSE / R2 | Pronostico de demanda con error tolerable para planificar compras (MAE bajo, R2 alto). El modelo avanzado debe ser al menos **competitivo** con el baseline adaptativo (media movil); por el caracter no estacionario de la serie, un buen baseline es dificil de superar y se documenta el analisis. |
| Salud (clasificacion) | F1 (clase Alto) y Accuracy | Minimizar falsos negativos de la clase Alto (no clasificar un paciente critico como Bajo). |

**Criterio critico de negocio:** en salud, el costo de un falso negativo (Alto -> Bajo) es inaceptable; se prioriza el *recall* de la clase Alto.

## 1.4. Alineacion con los ODS

- **ODS 3 (Salud y Bienestar):** deteccion temprana de pacientes prioritarios y disponibilidad garantizada de medicamentos oncologicos.
- **ODS 10 (Reduccion de Desigualdades):** se incorporan variables sociales (nivel socioeconomico, zona rural) para no dejar atras a los mas vulnerables.

In [1]:
import sys
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'src' / 'aldimi_common.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import aldimi_common as ac
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = (9, 5)
pd.set_option('display.max_columns', None)
print('Raiz del proyecto:', ROOT)

Raiz del proyecto: D:\2026-01\MachLearning\finTF


## 1.5. Datasets del proyecto

Se utilizan dos datasets reales de Kaggle, uno por frente.

In [2]:
print('FRENTE 2 - Salud   :', ac.KAGGLE_HEALTH_ID)
print('  archivo crudo    :', ac.HEALTH_RAW_FILE)
print('FRENTE 1 - Logistica:', ac.KAGGLE_STOCK_ID)
print('  archivo crudo    :', ac.STOCK_RAW_FILE)
print('\nObjetivo salud     :', ac.PRIORITY_TARGET, ac.PRIORITY_ORDER)
print('Objetivo logistica :', ac.DEMAND_TARGET_7, '/', ac.DEMAND_TARGET_14)

FRENTE 2 - Salud   : ankushpanday1/leukemia-cancer-risk-prediction-dataset
  archivo crudo    : biased_leukemia_dataset.csv
FRENTE 1 - Logistica: ziya07/high-dimensional-supply-chain-inventory-dataset
  archivo crudo    : supply_chain_dataset1.csv

Objetivo salud     : Prioridad_Atencion ['Bajo', 'Medio', 'Alto']
Objetivo logistica : Demanda_Fut_7d / Demanda_Fut_14d


## 1.6. Plan CRISP-DM

1. **Business Understanding** (este notebook).
2. **Data Understanding + EDA** (notebooks 02-04).
3. **Data Preparation + Integracion en BD** (notebooks 05-07).
4. **Modeling**: baseline (08) y avanzado en Colab (09-10).
5. **Evaluation** en Colab (11).
6. **Deployment**: dashboard `streamlit_app.py`.

> **Conclusion de la Fase 1:** el proyecto tiene objetivos de negocio claros, medibles y alineados con los ODS 3 y 10, con dos tareas de ML bien definidas.